# 04 — Temporal Audit & Candidate t0

**Objetivo:** reconstruir la semántica temporal y evaluar la viabilidad del timestamp de decisión.

Este notebook es un **laboratorio reproducible**. La lógica reutilizable vive en `src/argos/`;
aquí solo se parametriza, ejecuta, inspecciona evidencia y documentan decisiones.

## Preguntas
- ¿Qué fechas existen y qué cobertura tienen?
- ¿Qué rangos temporales cubre cada fuente?
- ¿Qué campos no parsean correctamente?
- ¿Se cumplen órdenes lógicos básicos de fechas?
- ¿`fecha_de_firma` es un `t0` operativo razonable o debe cambiarse?

Este notebook es crítico: una columna presente hoy no necesariamente existía en el momento histórico de predicción.

In [ ]:
from pathlib import Path
import tomllib
import pandas as pd

from argos.common.paths import ProjectPaths
from argos.config.registry import load_registry, discover_dataset_files

paths = ProjectPaths.discover()
paths.ensure_runtime_dirs()
DATASETS, RELATIONSHIPS = load_registry(paths.configs / "datasets.toml")
FILES = discover_dataset_files(paths.bronze, DATASETS)

with (paths.configs / "project.toml").open("rb") as handle:
    PROJECT_CONFIG = tomllib.load(handle)

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 120)

{k: p.name for k, p in FILES.items()}

In [ ]:
from argos.common.reports import write_report
from argos.profiling.temporal import (
    audit_all_temporal,
    audit_order_rule,
    temporal_format_distribution,
    temporal_parse_failures,
)

cfg = PROJECT_CONFIG["profiling"]
temporal_cfg = PROJECT_CONFIG["temporal"]

temporal_profile = audit_all_temporal(
    FILES,
    sample_size=cfg["csv_sample_size"],
    date_order=temporal_cfg["date_order"],
    plausible_year_min=temporal_cfg["plausible_year_min"],
    plausible_year_max=temporal_cfg["plausible_year_max"],
    temp_directory=paths.root / ".cache" / "duckdb",
)

write_report(
    temporal_profile,
    paths.reports / "profiling" / "temporal_profile.parquet",
)

temporal_profile.sort_values(["dataset", "column"])


In [ ]:
# Re-evaluate known ordering constraints with the explicit SECOP parser.
rules = [
    ("fecha_de_firma", "fecha_de_inicio_del_contrato"),
    ("fecha_de_inicio_del_contrato", "fecha_de_fin_del_contrato"),
    ("fecha_inicio_liquidacion", "fecha_fin_liquidacion"),
]

temporal_rules = pd.DataFrame([
    audit_order_rule(
        "contracts",
        FILES["contracts"],
        earlier,
        later,
        sample_size=cfg["csv_sample_size"],
        date_order=temporal_cfg["date_order"],
        temp_directory=paths.root / ".cache" / "duckdb",
    )
    for earlier, later in rules
])

write_report(
    temporal_rules,
    paths.reports / "profiling" / "temporal_rules.parquet",
)

temporal_rules


In [ ]:
# Gate evidence for t0 and the addition-event timestamp.
gate_temporal_columns = [
    ("contracts", "fecha_de_firma"),
    ("contracts", "fecha_de_inicio_del_contrato"),
    ("additions", "fecharegistro"),
    ("sanctions", "fecha_evento"),
]

temporal_gate_view = temporal_profile.loc[
    temporal_profile.apply(
        lambda row: (row["dataset"], row["normalized_column"]) in gate_temporal_columns,
        axis=1,
    ),
    [
        "dataset", "column", "coverage", "parse_success_rate_nonnull",
        "plausible_rate_parsed", "min_timestamp", "max_timestamp",
        "raw_min_timestamp", "raw_max_timestamp",
    ],
]

temporal_gate_view


## Decisión
No congeles `t0` por intuición. Documenta por qué el timestamp elegido representa el instante de decisión y qué información era realmente observable entonces.

In [ ]:
# Make parser behavior auditable for the two timestamps that define the target.
format_frames = [
    temporal_format_distribution(
        dataset,
        FILES[dataset],
        column,
        sample_size=cfg["csv_sample_size"],
        date_order=temporal_cfg["date_order"],
        temp_directory=paths.root / ".cache" / "duckdb",
    )
    for dataset, column in [
        ("contracts", "fecha_de_firma"),
        ("additions", "fecharegistro"),
    ]
]
temporal_formats = pd.concat(format_frames, ignore_index=True)
write_report(
    temporal_formats,
    paths.reports / "profiling" / "temporal_format_distribution.parquet",
)
temporal_formats


In [ ]:
failure_frames = [
    temporal_parse_failures(
        dataset,
        FILES[dataset],
        column,
        sample_size=cfg["csv_sample_size"],
        date_order=temporal_cfg["date_order"],
        limit=25,
        temp_directory=paths.root / ".cache" / "duckdb",
    )
    for dataset, column in [
        ("contracts", "fecha_de_firma"),
        ("additions", "fecharegistro"),
    ]
]
temporal_failures = pd.concat(failure_frames, ignore_index=True)
write_report(
    temporal_failures,
    paths.reports / "profiling" / "temporal_parse_failures.parquet",
)
temporal_failures
